### Data Ingestion Pipeline to Vector DB Pipeline


In [1]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Abhiram\AppData\Local\Temp\ipykernel_22856\329546744.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
d:\GitHub\LangChain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all pdfs
def process_pdfs(pdf_directory:str):
    """ Process all pdfs in a pdf directory"""
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    pdf_files=list(pdf_dir.glob('**/*.pdf'))
    print(f"found {len(pdf_files)}")

    for pdf in pdf_files:
        print(f'Processing: {pdf.name}')
        try:
            loader=PyMuPDFLoader(str(pdf))
            documents=loader.load()

            # Addtional Info (metadata)
            for doc in documents:
                doc.metadata['source_file']=pdf.name
                doc.metadata['file_type']='pdf'
                doc.metadata['author']='chatgpt'

            all_documents.extend(documents)
            print(f'Loaded {len(documents)} pages')

        except Exception as e:
            print("Error occured "+ ' '+ e)

    print(f'Total Documents: {len(all_documents)}')
    return all_documents

all_docs = process_pdfs('pdf/')
all_docs

found 3
Processing: apple.pdf
Loaded 11 pages
Processing: google.pdf
Loaded 11 pages
Processing: microsoft.pdf
Loaded 11 pages
Total Documents: 33


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-06-02T14:34:29+00:00', 'source': 'pdf\\apple.pdf', 'file_path': 'pdf\\apple.pdf', 'total_pages': 11, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': 'chatgpt', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-06-02T14:34:29+00:00', 'trapped': '', 'modDate': "D:20260602143429+00'00'", 'creationDate': "D:20260602143429+00'00'", 'page': 0, 'source_file': 'apple.pdf', 'file_type': 'pdf'}, page_content='Apple - RAG Test Knowledge Base\nCompany Overview\nApple: Detailed overview of the company, mission, vision, market position, and strategic direction.\nDetailed overview of the company, mission, vision, market position, and strategic direction. Detailed\noverview of the company, mission, vision, market position, and strategic direction. Detailed\noverview of the company, mission, vision, market position, and strategic direction. Detailed\noverview 

In [3]:
### Text Splitting (Chunking)
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n','\n',' ',''] 
    )

    split_docs=text_splitter.split_documents(documents)
    print(f'Split {len(documents)} documents into {len(split_docs)} chunks')
    
    if split_docs:
        print(f"Example Chunk")
        print(f"Content: {split_docs[0].page_content[:100]}....")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [4]:
chunks=split_documents(all_docs)


Split 33 documents into 187 chunks
Example Chunk
Content: Apple - RAG Test Knowledge Base
Company Overview
Apple: Detailed overview of the company, mission, v....
Metadata: {'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-06-02T14:34:29+00:00', 'source': 'pdf\\apple.pdf', 'file_path': 'pdf\\apple.pdf', 'total_pages': 11, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': 'chatgpt', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-06-02T14:34:29+00:00', 'trapped': '', 'modDate': "D:20260602143429+00'00'", 'creationDate': "D:20260602143429+00'00'", 'page': 0, 'source_file': 'apple.pdf', 'file_type': 'pdf'}


### Embeddings and Vector Store

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Tuple,Any
from sklearn.metrics.pairwise import cosine_similarity

In [40]:
class EmbeddingsManager:
    """Handles Document Embeddings generation using SentenceTransformers"""

    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """
          Initialize the embedding manager

        Args:
        model_name: HuggingFace model name for sentence embeddings
        """
        self._model_name=model_name
        self.model=None
        self._load_model()
    
    def _load_model(self):
        """Loads the Sentence Transformers Model"""
        try:
            print(f"Loading embedding model:{self._model_name}")
            self.model=SentenceTransformer(self._model_name)
            print(f"Model Loaded Successfully. Embedding dimensions: {self.model.get_embedding_dimension()}")
        
        except Exception as e:
            print(f"Error occured while model loading: {e}")
            raise
    
    def generate_embeddings(self,texts: List[str]) -> np.ndarray:
        """
        Generate embeddings.for a list of texts

        Args:
        texts: List of text strings to embed

        Returns:
        numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not Loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings



In [41]:
### Initialize Embeddings Manager

embedding_manager = EmbeddingsManager()
embedding_manager

Loading embedding model:all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10840.17it/s]


Model Loaded Successfully. Embedding dimensions: 384


In [50]:
### Vector Store

class VectorStore:
    """Manages document embeddings in ChromaDB Vector store"""

    def __init__(self,collection_name:str='pdf_documents',persist_directory:str='data/vector_store'):
        """Initialize the vector store

            Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()

    
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            # Create A Persistent ChromaDB Client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection =self.client.get_or_create_collection(name=self.collection_name,metadata={"description":"PDF document embeddings for RAG"})

            print(f"Vector Store initialized. Collection:{self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error initializing vector store :{e}")
            raise
    def delete_collection(self):
        """Delete the entire collection"""
        try:
            self.client.delete_collection(self.collection_name)

            print(f"Deleted collection: {self.collection_name}")

            # Recreate empty collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print("Created new empty collection")

        except Exception as e:
            print(f"Error deleting collection: {e}")
            raise

    def add_documents(self, documents, embeddings):

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())  # <- fixed

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

    


In [ ]:
vectorstore=VectorStore()
vectorstore

Vector Store initialized. Collection:pdf_documents
Existing documents in collection: 0


0

In [23]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the embeddings

embeddings=embedding_manager.generate_embeddings(texts=texts)

vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 187 texts...


Batches: 100%|██████████| 6/6 [00:03<00:00,  1.54it/s]


Generated embeddings with shape: (187, 384)
Successfully added 187 documents
Total documents in collection: 562


### RAG Retriever Pipeline


In [37]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self,vector_store:VectorStore,embeddings_manager:EmbeddingsManager):
        """
        Initialize the retriever
        Args:
        vector_store: Vector store containing document embeddings
        embedding_manager: Manager for generating query embeddings
        """
        self.vector_store=vector_store
        self.embeddings_manager=embeddings_manager
    
    def retrieve(self,query:str,top_k:int=5,score_threshold:float=0.0)->List[Dict[str,Any]]:
        """Retrieve relevant documents for a query

            Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

            Returns:
            List of dictionaries containing retrieved documents and metadata

            relevant documents for a query
        """
        print(f"Retreiving documents for query: {query}")
        print(f"Top K: {top_k}, score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding= self.embeddings_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            # print(results)
            print(self.vector_store.collection.count())
            retrieved_docs=[]
            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]

                for i, (doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)

                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                        print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
                    
                    else:
                        print("No related documents found")
                
                return retrieved_docs
        
        except Exception as e:
            print(f"Error during retreival: {e}")
            return []

In [38]:
rag_retriever = RAGRetriever(vectorstore,embedding_manager)
rag_retriever

In [39]:
print(rag_retriever.retrieve("What is attention all you need?"))

Retreiving documents for query: What is attention all you need?
Top K: 5, score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.22it/s]

Generated embeddings with shape: (1, 384)
562
No related documents found
No related documents found
No related documents found
No related documents found
No related documents found
[]
